In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [50]:
#new version old one was removing rows.
import pandas as pd
import os
from glob import glob

# Define directories
lat_dir = r"../output/Tohoku_foreshock/No_removal/RAW_data/PCA/N_MOM/After_pca79"
long_dir = r"../output/Tohoku_foreshock/No_removal/RAW_data/PCA/E_MOM/After_pca86"
up_dir = r"../output/Tohoku_foreshock/No_removal/RAW_data/PCA/U_MOM/After_pca51"
output_dir = r"../output/Tohoku_foreshock/No_removal/Combined/PCA"

# List all files in each directory
lat_files = glob(os.path.join(lat_dir, "*.mom"))
long_files = glob(os.path.join(long_dir, "*.mom"))
up_files = glob(os.path.join(up_dir, "*.mom"))

# Extract station names from filenames
lat_stations = set(os.path.basename(f).split('.')[0] for f in lat_files)
long_stations = set(os.path.basename(f).split('.')[0] for f in long_files)
up_stations = set(os.path.basename(f).split('.')[0] for f in up_files)

# Find common stations across all directories
common_stations = lat_stations & long_stations & up_stations

if not common_stations:
    print("No matching stations found across all directories.")
else:
    for station in common_stations:
        # Construct full file paths for this station
        lat_file = os.path.join(lat_dir, f"{station}.mom")
        long_file = os.path.join(long_dir, f"{station}.mom")
        up_file = os.path.join(up_dir, f"{station}.mom")

        try:
            # Read the files into pandas DataFrames without headers
            lat_data = pd.read_csv(lat_file, sep=r'\s+', header=None)
            long_data = pd.read_csv(long_file, sep=r'\s+', header=None)
            up_data = pd.read_csv(up_file, sep=r'\s+', header=None)

            # Assign column names
            lat_data.columns = ['time', 'lat']
            long_data.columns = ['time', 'long']
            up_data.columns = ['time', 'up']

            # Use the time column from lat_data
            time_column = lat_data['time']

            # Concatenate based on index instead of merging on "time"
            combined_df = pd.concat([time_column, lat_data['lat'], long_data['long'], up_data['up']], axis=1)

            # Rename columns properly
            combined_df.columns = ['time', 'lat', 'long', 'up']

            # Save to space-separated .dat file
            output_file = os.path.join(output_dir, f"{station}_combined.dat")
            combined_df.to_csv(output_file, index=False, header=False, sep=' ')

        except Exception as e:
            print(f"Error processing station {station}: {e}")

In [11]:
#updated to divide by 1000
import os

# Set directory paths
orig_dir = r"../prec/data_byEQ/Sanrika"   # Path to original directory
sep_dir = r"../output/Sanrika/Removal/Combined/ICA"   # Path to separate directory
output_dir = r"../output/Sanrika/Removal/Final/ICA"  # Path to save output files

# Make sure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Process the files
for original_filename in os.listdir(orig_dir):
    if original_filename.endswith('.dat') and original_filename.startswith('TS_'):
        # Extract the variable part of the filename (e.g., I026 from TS_I026.dat)
        base_filename = original_filename[3:-4]  # Remove 'TS_' and '.dat'

        # Determine the corresponding separate file name (e.g., I026_combined.dat)
        separate_filename = f"{base_filename}_combined.dat"
        original_file_path = os.path.join(orig_dir, original_filename)
        separate_file_path = os.path.join(sep_dir, separate_filename)

        if os.path.exists(separate_file_path):
            # Read the original and separate files
            with open(original_file_path, 'r') as original_file:
                original_lines = original_file.readlines()

            with open(separate_file_path, 'r') as separate_file:
                separate_lines = separate_file.readlines()

            # Process the lines
            updated_lines = []
            for orig_line, sep_line in zip(original_lines, separate_lines):
                # Split both lines using split() to handle multiple spaces/tabs
                orig_cols = orig_line.split()
                sep_cols = sep_line.split()

                # Replace columns 8, 9, and 10 in the original with columns 2, 3, and 4 from the separate file
                if len(orig_cols) >= 10 and len(sep_cols) >= 4:
                    # Divide columns 2, 3, and 4 from the separate file by 1000
                    sep_cols = [str(float(col) / 1000) for col in sep_cols[1:4]]

                    # Update the line with the modified values
                    updated_line = orig_cols[:7] + sep_cols + orig_cols[10:]  # Modify columns 8-10
                else:
                    updated_line = orig_line  # If the line is shorter than expected, leave it unchanged

                updated_lines.append(" ".join(updated_line) + "\n")

            # Write the updated lines to the output file
            output_file_path = os.path.join(output_dir, f"updated_{original_filename}")
            with open(output_file_path, 'w') as output_file:
                output_file.writelines(updated_lines)

        else:
            print(f"Separate file for {original_filename} not found.")